# BoardMind AI: Final Model Evaluation & Benchmark
**Notebook:** `08_whiteboard_final_evaluation.ipynb`  
**Author:** Senior Computer Vision & ML Research Engineer  
**Hardware:** Intel Core i9-13900H CPU, 16GB RAM, Windows  

### Objectives:
1. Systematically review validation metrics across all optimization experiments.
2. Select the optimal model based on generalization, mAP@50-95, F1 score, and CPU inference latency.
3. Save the champion checkpoint to `models/whiteboard_detector_best.pt`.
4. Evaluate the champion model **exactly once** on the untouched `test` split.
5. Provide production inference code for real-time classroom whiteboard detection in BoardMind AI.

In [ ]:
import sys
import os
import json
import shutil
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from evaluate_model import evaluate_checkpoint, benchmark_inference

DATA_YAML = PROJECT_ROOT / 'datasets' / 'whiteboard' / 'data.yaml'
REGISTRY_FILE = PROJECT_ROOT / 'runs' / 'whiteboard_optimization' / 'experiment_registry.json'
BEST_MODEL_TARGET = PROJECT_ROOT / 'models' / 'whiteboard_detector_best.pt'

## 1. Comprehensive Validation Comparison Table
Loading all experiment results from the central registry.

In [ ]:
with open(REGISTRY_FILE, 'r') as f:
    registry = json.load(f)

rows = []
for name, rec in registry.items():
    m = rec['metrics_val']
    lat = rec.get('latency_benchmark', {})
    rows.append({
        'Experiment': name,
        'Category': rec['category'],
        'Model': rec['model_source'],
        'Image Size': rec['imgsz'],
        'Precision': m['precision'],
        'Recall': m['recall'],
        'F1': m.get('f1_score', 0),
        'mAP50': m['mAP50'],
        'mAP50-95': m['mAP50_95'],
        'Latency (ms)': lat.get('mean_latency_ms', m.get('total_latency_ms', 0)),
        'FPS': lat.get('fps', m.get('fps', 0)),
        'Checkpoint': rec['best_checkpoint']
    })

df_comparison = pd.DataFrame(rows).sort_values(by='mAP50-95', ascending=False)
display(df_comparison)

## 2. Model Selection & Export to `models/whiteboard_detector_best.pt`

In [ ]:
# Select the top-performing model on validation set
best_exp_row = df_comparison.iloc[0]
best_ckpt_src = Path(best_exp_row['Checkpoint'])

print(f"Selected Champion Model: {best_exp_row['Experiment']}")
print(f"Source Path: {best_ckpt_src}")

# Export to models/whiteboard_detector_best.pt
BEST_MODEL_TARGET.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best_ckpt_src, BEST_MODEL_TARGET)
print(f"Successfully saved champion checkpoint to: {BEST_MODEL_TARGET}")

## 3. Final Untouched Test Set Evaluation (Executed Exactly Once)
In accordance with ML research rigor, the test set is evaluated only after model selection is finalized.

In [ ]:
test_results, test_metrics = evaluate_checkpoint(
    BEST_MODEL_TARGET,
    data_yaml=DATA_YAML,
    split='test',
    imgsz=int(best_exp_row['Image Size']),
    batch=4
)

print("=== UNTOUCHED TEST SET PERFORMANCE ===")
print(json.dumps(test_results, indent=2))

## 4. Production Inference Quick-Start
Code snippet demonstrating real-time classroom whiteboard detection in BoardMind AI.

In [ ]:
from ultralytics import YOLO
import cv2

# Load champion detector
detector = YOLO(str(BEST_MODEL_TARGET))

# Run inference on a sample frame
sample_frame = PROJECT_ROOT / 'notebooks' / 'board-srmap.jpg'
results = detector.predict(source=str(sample_frame), conf=0.25, imgsz=416, device='cpu')

for r in results:
    print(f"Detected {len(r.boxes)} whiteboard(s):")
    for i, box in enumerate(r.boxes):
        xyxy = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        print(f"  Board {i+1}: Coordinates [xmin, ymin, xmax, ymax] = {xyxy}, Confidence = {conf:.2f}")